# 🍐 Little Fig Studio

**Train any LLM on any hardware.** Select your model, configure, launch.

| Feature | Result |
|---|---|
| Quantization | Beats NF4 on 156/156 layers |
| GPU Speed | 7× faster than BnB NF4 |
| CPU Training | 1.1B model in 400MB RAM |
| Optimizer | FigMeZO (−18.6%, original research) |

**Run all cells below ↓**

In [9]:
# Step 1: Install Little Fig + dependencies
!git clone https://github.com/ticketguy/littlefig.git /content/littlefig 2>/dev/null || (cd /content/littlefig && git pull)

# Install with [train] extras (embers-diaries is not yet on PyPI)
!pip install -e "/content/littlefig[train]" -q
!pip install lm-eval -q


# Install cloudflared binary for tunnel
import subprocess, os
if not os.path.exists('/usr/local/bin/cloudflared'):
    subprocess.run(['wget', '-q', 'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64',
                    '-O', '/usr/local/bin/cloudflared'], check=True)
    os.chmod('/usr/local/bin/cloudflared', 0o755)
    print('✅ cloudflared installed')
else:
    print('✅ cloudflared already installed')

# Verify install
import sys, importlib
# Add source path and clear any cached import failures from prior runs
if '/content/littlefig/src' not in sys.path:
    sys.path.insert(0, '/content/littlefig/src')
# Remove any cached failed imports of little_fig
for key in list(sys.modules.keys()):
    if 'little_fig' in key:
        del sys.modules[key]
importlib.invalidate_caches()

import torch
from little_fig.engine import __version__ as fig_version
print(f'✅ Little Fig v{fig_version}')
print(f'✅ PyTorch {torch.__version__} | CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'✅ GPU: {torch.cuda.get_device_name(0)}')

remote: Enumerating objects: 25, done.
remote: Counting objects: 100% (25/25), done.
remote: Compressing objects: 100% (5/5), done.
remote: Total 17 (delta 9), reused 16 (delta 8), pack-reused 0 (from 0)
Unpacking objects: 100% (17/17), 13.73 KiB | 265.00 KiB/s, done.
From https://github.com/ticketguy/littlefig
   1d67b76..c9cc5c8  main       -> origin/main
Updating 1d67b76..c9cc5c8
Fast-forward
 pyproject.toml                       |   6 +
 src/little_fig/web/server.py         | 570 +++++++++++++++++++++++++++++++++--
 src/little_fig/web/static/index.html | 265 ++++++++++++++--
 3 files changed, 800 insertions(+), 41 deletions(-)
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for little-fig (pyproject.toml) ... done
✅ cloudflared already installed
✅ Little Fig v0.6.0
✅ PyTorch 2.10.0+cu128 | CUDA: True
✅ GP

In [8]:
# Step 2: Launch Little Fig Studio
import subprocess, time, re, sys, os
sys.path.insert(0, '/content/littlefig/src')

# Kill any previous server/tunnel
!kill $(lsof -t -i:8888) 2>/dev/null; pkill -f cloudflared 2>/dev/null; true
time.sleep(1)

# Start the web server
server_log = open('/tmp/fig_server.log', 'w')
server = subprocess.Popen(
    [sys.executable, '-m', 'uvicorn', 'little_fig.web.server:app',
     '--host', '0.0.0.0', '--port', '8888'],
    cwd='/content/littlefig/src',
    stdout=server_log, stderr=server_log
)
time.sleep(4)

# Check if server started
import urllib.request
server_ok = False
for attempt in range(3):
    try:
        urllib.request.urlopen('http://localhost:8888/api/health', timeout=3)
        server_ok = True
        break
    except:
        time.sleep(2)

if not server_ok:
    print('❌ Server failed to start. Log:')
    with open('/tmp/fig_server.log') as f:
        print(f.read()[-1000:])
else:
    print('✅ Server running on port 8888')

    # Start cloudflared tunnel — write output to file for reliable reading
    tunnel_log = open('/tmp/fig_tunnel.log', 'w')
    tunnel = subprocess.Popen(
        ['cloudflared', 'tunnel', '--url', 'http://localhost:8888'],
        stdout=tunnel_log, stderr=tunnel_log
    )

    # Poll the log file for the tunnel URL (takes 3-10 seconds)
    tunnel_url = None
    for i in range(20):
        time.sleep(1)
        with open('/tmp/fig_tunnel.log') as f:
            text = f.read()
        urls = re.findall(r'https://[a-z0-9-]+\.trycloudflare\.com', text)
        if urls:
            tunnel_url = urls[0]
            break

    if tunnel_url:
        from IPython.display import display, HTML
        print(f'\n🍐 Little Fig Studio is LIVE!')
        print(f'\n   👉 {tunnel_url}')
        print(f'\n   Open the link above in your browser.')
        print(f'   Select model → Configure → Train')
        print(f'\n   Keep this cell running to keep the server alive.')
        display(HTML(f'<h3><a href="{tunnel_url}" target="_blank">🍐 Open Little Fig Studio</a></h3>'))
    else:
        print('\n⚠️ Cloudflare tunnel did not start. Log:')
        with open('/tmp/fig_tunnel.log') as f:
            print(f.read()[-500:])
        print('\n   Trying Colab iframe instead...')
        from IPython.display import display, IFrame
        display(IFrame(src='http://localhost:8888', width='100%', height=700))

^C
✅ Server running on port 8888

🍐 Little Fig Studio is LIVE!

   👉 https://ind-roy-funny-seventh.trycloudflare.com

   Open the link above in your browser.
   Select model → Configure → Train

   Keep this cell running to keep the server alive.


---
## Alternative: Python API (no UI)

Change `MODEL` to any HuggingFace model.

In [ ]:
import sys
sys.path.insert(0, '/content/littlefig/src')
from little_fig.engine import FigModel, FigTrainer, FigTrainingConfig

# === YOUR MODEL HERE ===
MODEL = 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'
# MODEL = 'google/gemma-3-4b-it'
# MODEL = 'Qwen/Qwen2.5-1.5B'
# MODEL = 'microsoft/phi-2'

model = FigModel.from_pretrained(MODEL, lora_r=16, lora_alpha=32, shared_codebook=True)
print(f'✅ {MODEL} loaded | Trainable: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')

In [ ]:
# Train
config = FigTrainingConfig(
    num_epochs=1,
    learning_rate=2e-4,
    max_seq_length=256,
    batch_size=2,
    gradient_accumulation_steps=4,
    logging_steps=5,
    use_packing=True,
)
trainer = FigTrainer(model, config)
trainer.load_dataset('tatsu-lab/alpaca', max_samples=200)
trainer.train()
model.save_adapter('./my_adapter')
print('\n✅ Training complete! Adapter saved.')

---
*0xticketguy / Harboria Labs | AGPL-3.0*